# 🔍 Thử nghiệm Entity Extraction từ câu hỏi Giao thông
**Mục tiêu:** Thiết kế và kiểm tra một prompt dùng Gemini để tự động trích xuất các thực thể quan trọng từ câu hỏi của người dùng.

Các thực thể cần trích xuất:
- `loai_xe`: Loại phương tiện (xe máy, ô tô, xe đạp, xe buýt...)
- `loai_vi_pham`: Hành vi vi phạm (nồng độ cồn, tốc độ, đèn đỏ...)
- `doi_tuong_vi_pham`: Người thực hiện (người lái, chủ xe, hành khách...)
- `so_tien_phat`: Số tiền đề cập trong câu (nếu có)

**Ý nghĩa thực tiễn:** Dùng entity để lọc (filter) trước khi tìm kiếm vector → tăng độ chính xác.

In [ ]:
import os, sys, json
from dotenv import load_dotenv

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

import google.generativeai as genai
from source.core.config import Settings

settings = Settings()
api_key = settings.api_key or os.getenv('API_KEY')
genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-2.0-flash')
print("✅ Gemini sẵn sàng!")

## 1. Định nghĩa Prompt Entity Extraction

In [ ]:
ENTITY_PROMPT = """
Bạn là chuyên gia phân tích ngôn ngữ pháp luật giao thông Việt Nam.
Từ câu hỏi của người dùng, hãy trích xuất các thực thể pháp lý quan trọng.

Câu hỏi: "{query}"

Trả về JSON theo format sau (null nếu không có thông tin):
{{
  "loai_xe": null,                  // VD: "xe máy", "ô tô", "xe đạp", "xe tải"
  "loai_vi_pham": null,             // VD: "nồng độ cồn", "vượt tốc độ", "vượt đèn đỏ", "không đội mũ bảo hiểm"
  "doi_tuong": null,                // VD: "người điều khiển", "chủ xe", "hành khách"
  "moc_toc_do": null,               // VD: "quá 15km/h", "trên 60km/h" (nếu câu hỏi có đề cập)
  "moc_nong_do_con": null,          // VD: "vượt 0.25mg/l", "dưới 50mg/100ml"
  "van_ban_lien_quan": null,        // VD: "Nghị định 100", "Luật ATGT 2024"
  "chu_de_phap_ly": null            // VD: "xử phạt nồng độ cồn", "giấy phép lái xe"
}}

Chỉ trả về JSON, không giải thích.
"""

def extract_entities(query: str) -> dict:
    try:
        prompt = ENTITY_PROMPT.format(query=query)
        response = model.generate_content(prompt)
        text = response.text.strip().replace('```json', '').replace('```', '').strip()
        return json.loads(text)
    except Exception as e:
        return {"error": str(e)}

# Test nhanh
test_q = "tôi đi xe máy uống 2 chai bia, bị thổi nồng độ cồn 0.3mg/l thì phạt bao nhiêu?"
result = extract_entities(test_q)
print(f"Q: {test_q}")
print(json.dumps(result, ensure_ascii=False, indent=2))

## 2. Chạy thử nghiệm trên nhiều câu hỏi

In [ ]:
TEST_QUERIES = [
    "xe máy vượt đèn đỏ tại ngã tư bị phạt bao nhiêu tiền?",
    "ô tô không thắt dây an toàn, phạt tiền thế nào?",
    "người lái xe tải chạy quá tốc độ 20km/h trên đường quốc lộ bị phạt gì?",
    "không có bảo hiểm bắt buộc TNDS thì bị phạt mấy triệu?",
    "xe đạp điện đi vào đường cao tốc có bị phạt không?",
    "lái xe ban đêm không có đèn chiếu sáng bị xử lý thế nào?",
]

print(f"🚀 Đang chạy Entity Extraction trên {len(TEST_QUERIES)} câu hỏi...\n")
all_results = []
for i, query in enumerate(TEST_QUERIES, 1):
    print(f"[{i}/{len(TEST_QUERIES)}] {query}")
    entities = extract_entities(query)
    all_results.append({"query": query, "entities": entities})
    
    # In tóm tắt
    loai_xe = entities.get('loai_xe', 'N/A')
    vi_pham = entities.get('loai_vi_pham', 'N/A')
    chu_de = entities.get('chu_de_phap_ly', 'N/A')
    print(f"   → Xe: {loai_xe} | Vi phạm: {vi_pham} | Chủ đề: {chu_de}")
    print()

## 3. Phân tích kết quả & Thống kê

In [ ]:
from collections import Counter

# Thống kê
loai_xe_counts = Counter()
vi_pham_counts = Counter()
null_counts = Counter()

for r in all_results:
    entities = r['entities']
    if entities.get('loai_xe'):
        loai_xe_counts[entities['loai_xe']] += 1
    if entities.get('loai_vi_pham'):
        vi_pham_counts[entities['loai_vi_pham']] += 1
    # Đếm số trường null
    null_count = sum(1 for v in entities.values() if v is None)
    null_counts[null_count] += 1

print("📊 THỐNG KÊ KẾT QUẢ ENTITY EXTRACTION:")
print(f"\n🚗 Loại xe phổ biến: {dict(loai_xe_counts.most_common())}")
print(f"⚡ Loại vi phạm phổ biến: {dict(vi_pham_counts.most_common())}")

# Đánh giá mức độ đầy đủ thông tin
total_fields = 7  # Tổng số trường trong schema
avg_null = sum(k*v for k,v in null_counts.items()) / len(all_results)
completeness = 1 - (avg_null / total_fields)
print(f"\n✨ Độ đầy đủ thông tin trung bình: {completeness:.1%}")
print(f"💡 Tỷ lệ trường NULL trung bình: {avg_null:.1f}/{total_fields}")

## 4. Demo: Dùng Entity để tăng cường câu truy vấn

In [ ]:
def entity_enhanced_query(original_query: str) -> str:
    """Dùng entities để tạo câu truy vấn tìm kiếm phong phú hơn."""
    entities = extract_entities(original_query)
    
    parts = [original_query]
    if entities.get('loai_xe'):
        parts.append(entities['loai_xe'])
    if entities.get('loai_vi_pham'):
        parts.append(entities['loai_vi_pham'])
    if entities.get('van_ban_lien_quan'):
        parts.append(entities['van_ban_lien_quan'])
    if entities.get('chu_de_phap_ly'):
        parts.append(entities['chu_de_phap_ly'])
    
    enhanced = ' | '.join(parts)
    return enhanced, entities

demo_query = "người đi SH uống ít bia bị dừng xe thổi cồn thì phạt bao nhiêu?"
enhanced, entities = entity_enhanced_query(demo_query)

print(f"📌 Câu gốc   : {demo_query}")
print(f"🔧 Entities  : {json.dumps(entities, ensure_ascii=False)}")
print(f"✨ Câu tăng cường: {enhanced}")
print("\n💡 Câu truy vấn Enhanced này sẽ được dùng để tìm kiếm trong Qdrant thay vì câu gốc!")